## NOAA flood data 1996-most recent for 10 states

In [3]:
import pandas as pd
import requests
import re
import os

In [4]:
base_url = (
    "https://www.ncei.noaa.gov/pub/data/swdi/"
    "stormevents/csvfiles/"
)

states = [
    "WEST VIRGINIA",
    "ALABAMA",
    "ARKANSAS",
    "IOWA",
    "KENTUCKY",
    "MISSISSIPPI",
    "MISSOURI",
    "NEBRASKA",
    "OKLAHOMA",
    "TENNESSEE"
]

response = requests.get(base_url, timeout=60)
response.raise_for_status()

files = re.findall(
    r'href="(StormEvents_details-ftp_v1\.0_d\d{4}_c\d+\.csv\.gz)"',
    response.text
)

year_files = {}

for filename in files:
    year = int(re.search(r"_d(\d{4})_", filename).group(1))
    if year >= 1996:
        year_files[year] = filename

print("Available years:", min(year_files), "-", max(year_files))
print("Files to download:", len(year_files))

flood_data = []

for year, filename in sorted(year_files.items()):

    url = base_url + filename
    print(f"Downloading and filtering {year}...")

    yearly_data = pd.read_csv(
        url,
        compression="gzip",
        low_memory=False
    )

    filtered = yearly_data[
        yearly_data["STATE"].str.strip().str.upper().isin(states) &
        yearly_data["EVENT_TYPE"].str.contains(
            "FLOOD", case=False, na=False
        )
    ].copy()

    flood_data.append(filtered)

noaa_floods = pd.concat(
    flood_data,
    ignore_index=True
)

noaa_floods = noaa_floods.sort_values(
    ["BEGIN_DATE_TIME", "STATE"]
).reset_index(drop=True)

os.makedirs("data/intermediate", exist_ok=True)

output_file = (
    "data/intermediate/NOAA_Flood_Events_10_States_1996_2026.csv"
)

noaa_floods.to_csv(output_file, index=False)

# 
print("\nDOWNLOAD COMPLETE")
print("Total flood records:", len(noaa_floods))
print("States:", noaa_floods["STATE"].nunique())
print("Years:", noaa_floods["YEAR"].min(), "-", noaa_floods["YEAR"].max())

print("\nRecords by state:")
display(
    noaa_floods.groupby("STATE").size()
    .sort_values(ascending=False)
    .to_frame("flood_records")
)

print("\nRecords by event type:")
display(
    noaa_floods.groupby("EVENT_TYPE").size()
    .sort_values(ascending=False)
    .to_frame("records")
)

print("\nSaved to:", output_file)


Available years: 1996 - 2026
Files to download: 31

DOWNLOAD COMPLETE
Total flood records: 54898
States: 10
Years: 1996 - 2026

Records by state:


,flood_records
STATE,
MISSOURI,10830
KENTUCKY,8324
IOWA,7389
ARKANSAS,5693
TENNESSEE,4493
OKLAHOMA,4347
WEST VIRGINIA,4243
MISSISSIPPI,3820
ALABAMA,3389



Records by event type:


,records
EVENT_TYPE,
Flash Flood,34131
Flood,20710
Coastal Flood,57



Saved to: data/raw/NOAA_Flood_Events_10_States_1996_2026.csv


### now have NOAA data saved to data/intermediate for the 10 states called NOAA_Flood_Events_10_States_1996_2026.csv

#### Figure out the best way to aggregate it spatially? Pull in census api to amend county boundaries to this output and export it as a geodataframe